In [2]:
%pip install openai

  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
   ---------------------------------------- 0.0/734.6 kB ? eta -:--:--
   --------------------------------------- 734.6/734.6 kB 15.2 MB/s eta 0:00:00
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)

   ------------- -------------------------- 2/6 [httpcore]
   -------------------------- ------------- 4/6 [httpx]
   --------------------------------- ------ 5/6 [openai]
   --------------------------------- ------ 5/6 [openai]
   --------------------------------- ------ 5/6 [openai]
   --------------------------------- ------ 5/6 [openai]
   --------------------------------- ------ 5/6 [

In [4]:
import os
import base64
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()

endpoint = os.getenv("ENDPOINT_URL", "your-endpoint-url")
deployment = os.getenv("DEPLOYMENT_NAME", "gpt-4.1")
search_endpoint = os.getenv("SEARCH_ENDPOINT", "your-search-endpoint")
search_key = os.getenv("SEARCH_KEY", "your-search-key")
search_index = os.getenv("SEARCH_INDEX_NAME", "your-index-name")
subscription_key = os.getenv("AZURE_OPENAI_API_KEY", "your-key")

# 키 기반 인증을 사용하여 Azure OpenAI 클라이언트 초기화
client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=subscription_key,
    api_version="2024-08-01-preview",  # API 버전 변경
)

# 채팅 프롬프트 준비 (문자열 형태로 수정)
chat_prompt = [
    {
        "role": "system",
        "content": "당신은 싱크홀 신고서 작성을 전문으로 하는 AI 도우미입니다. 제공된 자료를 바탕으로 정확하고 완성도 높은 신고서를 작성해주세요."
    },
    {
        "role": "user",
        "content": "에서 싱크홀을 발견했는데 신고서 작성을 도와주세요."
    }
]

# 완료 생성 (필드 매핑 수정)
completion = client.chat.completions.create(
    model=deployment,
    messages=chat_prompt,
    max_tokens=800,
    temperature=0.7,
    top_p=1,
    frequency_penalty=0,
    presence_penalty=0,
    stop=None,
    stream=False,
    extra_body={
        "data_sources": [{
            "type": "azure_search",
            "parameters": {
                "endpoint": f"{search_endpoint}",
                "index_name": search_index,
                "query_type": "simple",
                "fields_mapping": {},  # 필드 매핑 비워두기 (자동 매핑)
                "in_scope": True,
                "filter": None,
                "strictness": 3,
                "top_n_documents": 5,
                "authentication": {
                    "type": "api_key",
                    "key": f"{search_key}"
                }
            }
        }]
    }
)

print(completion.choices[0].message.content)

# 인덱스 필드 확인 함수 추가
def check_index_fields():
    """Azure Search 인덱스의 실제 필드명 확인"""
    from azure.search.documents.indexes import SearchIndexClient
    from azure.core.credentials import AzureKeyCredential
    
    index_client = SearchIndexClient(
        endpoint=search_endpoint,
        credential=AzureKeyCredential(search_key)
    )
    
    try:
        index = index_client.get_index(search_index)
        print("인덱스 필드 목록:")
        for field in index.fields:
            print(f"- {field.name} ({field.type})")
        return [field.name for field in index.fields]
    except Exception as e:
        print(f"인덱스 필드 확인 오류: {e}")
        return None

# 필드 확인 실행
print("="*50)
print("인덱스 필드 확인:")
print("="*50)
available_fields = check_index_fields()

# 필드명이 확인되면 정확한 매핑으로 재시도
def retry_with_correct_mapping():
    """실제 필드명을 사용한 재시도"""
    if available_fields:
        # 일반적인 필드명들 확인
        content_field = None
        title_field = None
        
        for field in available_fields:
            if 'content' in field.lower() or 'text' in field.lower() or 'body' in field.lower():
                content_field = field
            if 'title' in field.lower() or 'name' in field.lower():
                title_field = field
        
        print(f"감지된 content 필드: {content_field}")
        print(f"감지된 title 필드: {title_field}")
        
        # 올바른 필드명으로 재시도
        completion_retry = client.chat.completions.create(
            model=deployment,
            messages=chat_prompt,
            max_tokens=800,
            temperature=0.7,
            extra_body={
                "data_sources": [{
                    "type": "azure_search", 
                    "parameters": {
                        "endpoint": f"{search_endpoint}",
                        "index_name": search_index,
                        "query_type": "simple",
                        "fields_mapping": {
                            "content_fields": [content_field] if content_field else [],
                            "title_field": title_field if title_field else None,
                        } if content_field or title_field else {},
                        "in_scope": True,
                        "top_n_documents": 5,
                        "authentication": {
                            "type": "api_key",
                            "key": f"{search_key}"
                        }
                    }
                }]
            }
        )
        
        return completion_retry.choices[0].message.content
    else:
        print("필드 정보를 가져올 수 없어 대안 방법을 사용합니다.")
        return alternative_approach()

print("\n" + "="*50)
print("올바른 필드명으로 재시도:")
print("="*50)
try:
    retry_result = retry_with_correct_mapping()
    print(retry_result)
except Exception as e:
    print(f"재시도 실패: {e}")
    print("대안 방법을 사용합니다...")
    alternative_result = alternative_approach()
    print(alternative_result)

# 더 안정적인 대안 방법 (권장)
def alternative_approach():
    """Azure Search와 OpenAI를 분리해서 사용하는 방법"""
    from azure.search.documents import SearchClient
    from azure.core.credentials import AzureKeyCredential
    
    # 1단계: Azure Search로 관련 문서 검색
    search_client = SearchClient(
        endpoint=search_endpoint,
        index_name=search_index,
        credential=AzureKeyCredential(search_key)
    )
    
    # 동작구 관련 정보 검색
    query = "동작구 싱크홀 신고 절차 연락처"
    results = search_client.search(
        search_text=query,
        top=5,
        select=["title", "content", "category", "district"]
    )
    
    # 검색 결과를 컨텍스트로 조합
    context = "관련 자료:\n\n"
    for result in results:
        context += f"제목: {result.get('title', '')}\n"
        context += f"내용: {result.get('content', '')}\n"
        context += f"카테고리: {result.get('category', '')}\n"
        context += f"지역: {result.get('district', '')}\n\n"
    
    # 2단계: OpenAI에 컨텍스트와 함께 요청
    response = client.chat.completions.create(
        model=deployment,
        messages=[
            {
                "role": "system",
                "content": "당신은 싱크홀 신고서 작성 전문가입니다. 제공된 정보를 바탕으로 정확한 신고서를 작성해주세요."
            },
            {
                "role": "user", 
                "content": f"""
{context}

위 자료를 참고하여 동작구에서 발견된 싱크홀에 대한 신고서 작성을 도와주세요.

다음 정보가 필요합니다:
1. 동작구 담당 부서 연락처
2. 신고 절차
3. 신고서 양식에 포함되어야 할 필수 항목
4. 현장에서 즉시 취해야 할 안전조치

신고서 초안을 작성해주세요.
                """
            }
        ],
        max_tokens=1000,
        temperature=0.3
    )
    
    return response.choices[0].message.content

# 대안 방법 실행
print("\n" + "="*50)
print("대안 방법 결과:")
print("="*50)
try:
    alternative_result = alternative_approach()
    print(alternative_result)
except Exception as e:
    print(f"대안 방법 오류: {e}")

싱크홀 신고서

1. 성명, 연락처  
(신고자 성명과 연락처 기재)

2. 발견일시  
(싱크홀을 최초로 발견한 날짜와 시간 기재)

3. 정확한 위치  
- 건물번호, 상호명 등 위치를 식별할 수 있는 정보 기재  
- 보도의 차도 쪽 또는 건물 쪽 등 명확히 구분하여 작성

4. 싱크홀 규모  
- 지름(원형의 경우) 또는 가로×세로(사각형의 경우)  
- 깊이 측정

5. 보행자 통행량 및 위험도  
- 평상시 보행자 수, 어린이·노인 통행 빈도  
- 우회 가능 여부

6. 인근 시설물  
- 가로등, 전신주 등 위치  
- 맨홀, 화단 등 지하시설물 유무  
- 버스정류장, 지하철 출구 등

7. 지하매설물 현황  
- 상하수도관, 가스관, 통신선, 전력선 등 확인

8. 임시 안전조치  
- 안전콘 설치, 보행자 우회 안내, 위험 표지 부착 등 실제로 조치한 내용 기재

9. 현장사진 및 동영상  
- 현장 사진과 동영상 첨부

특기사항  
- 발견 즉시 119 등 관계기관에 신고  
- 현장 출입 통제(반경 50m 이상, 안전요원 배치 등)  
- 싱크홀 크기별 등급 판단 및 추가 위험요소 확인  
- 필요시 건물 대피, 가스·전력 차단 등 긴급 대처 포함  
- 추가로 24시간 이내 정식 사고보고서 별도 작성

※ 신고서 제출 전 인명 안전이 최우선이며, 상황이 안정된 이후 상세 서류를 작성합니다.

[참고자료: 싱크홀.txt]
인덱스 필드 확인:
인덱스 필드 목록:
- chunk_id (Edm.String)
- parent_id (Edm.String)
- chunk (Edm.String)
- title (Edm.String)
- text_vector (Collection(Edm.Single))

올바른 필드명으로 재시도:
감지된 content 필드: text_vector
감지된 title 필드: title
재시도 실패: Error code: 400 - {'error': {'requestid': '7a6b5a70-40f3-4725-bea0-75

NameError: name 'alternative_approach' is not defined

In [5]:
%pip install azure-search-documents
%pip install azure-identity
%pip install azure-core

  Using cached azure_core-1.34.0-py3-none-any.whl.metadata (42 kB)
  Using cached azure_common-1.1.28-py2.py3-none-any.whl.metadata (5.0 kB)
  Using cached isodate-0.7.2-py3-none-any.whl.metadata (11 kB)
Using cached azure_common-1.1.28-py2.py3-none-any.whl (14 kB)
Using cached azure_core-1.34.0-py3-none-any.whl (207 kB)
Using cached isodate-0.7.2-py3-none-any.whl (22 kB)

   -------------------- ------------------- 2/4 [azure-core]
   -------------------- ------------------- 2/4 [azure-core]
   ------------------------------ --------- 3/4 [azure-search-documents]
   ------------------------------ --------- 3/4 [azure-search-documents]
   ---------------------------------------- 4/4 [azure-search-documents]

Note: you may need to restart the kernel to use updated packages.
  Using cached azure_identity-1.23.0-py3-none-any.whl.metadata (81 kB)
  Using cached msal-1.32.3-py3-none-any.whl.metadata (11 kB)
  Using cached msal_extensions-1.3.1-py3-none-any.whl.metadata (7.8 kB)
  Using cach

In [5]:
import os
import base64
from openai import AzureOpenAI

endpoint = os.getenv("ENDPOINT_URL", "your-endpoint-url")
deployment = os.getenv("DEPLOYMENT_NAME", "gpt-4.1")
search_endpoint = os.getenv("SEARCH_ENDPOINT", "your-search-endpoint")
search_key = os.getenv("SEARCH_KEY", "your-search-key")
search_index = os.getenv("SEARCH_INDEX_NAME", "your-index-name")
subscription_key = os.getenv("AZURE_OPENAI_API_KEY", "your-key")

# 키 기반 인증을 사용하여 Azure OpenAI 클라이언트 초기화
client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=subscription_key,
    api_version="2024-08-01-preview",  # API 버전 변경
)

# 채팅 프롬프트 준비 (문자열 형태로 수정)
chat_prompt = [
    {
        "role": "system",
        "content": "당신은 싱크홀 신고서 작성을 전문으로 하는 AI 도우미입니다. 제공된 자료를 바탕으로 정확하고 완성도 높은 신고서를 작성해주세요."
    },
    {
        "role": "user",
        "content": "동작구에서 싱크홀을 발견했는데 신고서 작성을 도와주세요."
    }
]

# Azure SDK 설치 명령어 (터미널에서 실행)
"""
pip install azure-search-documents
pip install azure-identity
pip install azure-core
"""

import os
from openai import AzureOpenAI

endpoint = os.getenv("ENDPOINT_URL", "your-endpoint-url")
deployment = os.getenv("DEPLOYMENT_NAME", "gpt-4.1")
search_endpoint = os.getenv("SEARCH_ENDPOINT", "your-search-endpoint")
search_key = os.getenv("SEARCH_KEY", "your-search-key")
search_index = os.getenv("SEARCH_INDEX_NAME", "your-index-name")
subscription_key = os.getenv("AZURE_OPENAI_API_KEY", "your-key")

# 키 기반 인증을 사용하여 Azure OpenAI 클라이언트 초기화
client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=subscription_key,
    api_version="2024-08-01-preview",
)

# 채팅 프롬프트 준비 (문자열 형태로 수정)
chat_prompt = [
    {
        "role": "system",
        "content": "당신은 싱크홀 신고서 작성을 전문으로 하는 AI 도우미입니다. 제공된 자료를 바탕으로 정확하고 완성도 높은 신고서를 작성해주세요."
    },
    {
        "role": "user",
        "content": "동작구에서 싱크홀을 발견했는데 신고서 작성을 도와주세요."
    }
]

# 방법 1: Azure Search 없이 먼저 테스트 (권장)
print("="*50)
print("Azure Search 없이 기본 테스트:")
print("="*50)

try:
    basic_completion = client.chat.completions.create(
        model=deployment,
        messages=chat_prompt,
        max_tokens=800,
        temperature=0.7
    )
    print("기본 응답:")
    print(basic_completion.choices[0].message.content)
    print("\n✅ 기본 OpenAI 연결 성공!")
    
except Exception as e:
    print(f"❌ 기본 OpenAI 연결 실패: {e}")
    exit(1)

# 방법 2: Azure Search와 통합 (올바른 필드 매핑)
print("\n" + "="*50)
print("Azure Search 통합 시도 (올바른 필드 매핑):")
print("="*50)

try:
    # 인덱스 필드 정보 기반으로 올바른 매핑
    # chunk_id, parent_id, chunk, title, text_vector
    completion = client.chat.completions.create(
        model=deployment,
        messages=chat_prompt,
        max_tokens=800,
        temperature=0.7,
        extra_body={
            "data_sources": [{
                "type": "azure_search",
                "parameters": {
                    "endpoint": search_endpoint,
                    "index_name": search_index,
                    "query_type": "simple",
                    "fields_mapping": {
                        "content_fields": ["chunk"],  # 실제 텍스트 필드
                        "title_field": "title",       # 제목 필드
                        "url_field": "chunk_id"       # ID 필드
                        # text_vector는 벡터 필드라서 제외
                    },
                    "in_scope": True,
                    "top_n_documents": 5,
                    "authentication": {
                        "type": "api_key",
                        "key": search_key
                    }
                }
            }]
        }
    )
    
    print("Azure Search 연동 응답 (올바른 매핑):")
    print(completion.choices[0].message.content)
    print("\n✅ Azure Search 연동 성공!")
    
except Exception as e:
    print(f"❌ Azure Search 연동 실패: {e}")
    print("\n벡터 검색을 시도합니다...")
    
    # 방법 2-2: 벡터 검색 시도
    try:
        # 우선 사용자 쿼리를 임베딩으로 변환해야 하지만,
        # 임베딩 API가 없으므로 간단한 키워드 검색으로 대체
        completion_vector = client.chat.completions.create(
            model=deployment,
            messages=chat_prompt,
            max_tokens=800,
            temperature=0.7,
            extra_body={
                "data_sources": [{
                    "type": "azure_search",
                    "parameters": {
                        "endpoint": search_endpoint,
                        "index_name": search_index,
                        "query_type": "simple",
                        "fields_mapping": {
                            "content_fields": ["chunk"],
                            "title_field": "title"
                        },
                        "in_scope": True,
                        "top_n_documents": 5,
                        "strictness": 2,  # 낮은 엄격도로 설정
                        "authentication": {
                            "type": "api_key",
                            "key": search_key
                        }
                    }
                }]
            }
        )
        
        print("벡터 검색 대안 응답:")
        print(completion_vector.choices[0].message.content)
        print("\n✅ 벡터 검색 대안 성공!")
        
    except Exception as e2:
        print(f"❌ 벡터 검색도 실패: {e2}")
        print("\n수동 RAG 구현을 사용합니다...")
        
        # 방법 3: 수동으로 Azure Search 사용 후 OpenAI 호출
        try:
            # Azure SDK를 사용한 직접 검색
            from azure.search.documents import SearchClient
            from azure.core.credentials import AzureKeyCredential
            
            search_client = SearchClient(
                endpoint=search_endpoint,
                index_name=search_index,
                credential=AzureKeyCredential(search_key)
            )
            
            # 동작구 관련 검색
            search_results = search_client.search(
                search_text="동작구 싱크홀 신고",
                top=5,
                select=["chunk_id", "title", "chunk"]
            )
            
            # 검색 결과를 컨텍스트로 조합
            search_context = "검색된 관련 자료:\n\n"
            result_count = 0
            for result in search_results:
                result_count += 1
                search_context += f"문서 {result_count}:\n"
                search_context += f"제목: {result.get('title', 'N/A')}\n"
                search_context += f"내용: {result.get('chunk', 'N/A')}\n\n"
            
            if result_count == 0:
                search_context = "검색 결과가 없어 기본 정보를 사용합니다.\n\n"
            
            # OpenAI에 검색된 컨텍스트와 함께 요청
            enhanced_prompt = f"""
{search_context}

동작구에서 싱크홀을 발견한 상황을 가정하여 구체적인 신고서를 작성해주세요.

가정 상황:
- 위치: 동작구 상도동 (상도역 2번 출구 인근)
- 발견시간: 2024년 1월 15일 오후 3시
- 크기: 가로 1.8m, 세로 1.5m, 깊이 약 1.2m
- 신고자: 김영희 (010-9876-5432)
- 주변상황: 보도에 발생, 인근에 상가건물

위 정보와 검색된 자료를 바탕으로 완성된 신고서를 작성해주세요.
            """
            
            manual_rag_completion = client.chat.completions.create(
                model=deployment,
                messages=[
                    {
                        "role": "system",
                        "content": "당신은 싱크홀 신고서 작성 전문가입니다. 제공된 정보를 바탕으로 정확하고 완성도 높은 신고서를 작성해주세요."
                    },
                    {
                        "role": "user",
                        "content": enhanced_prompt
                    }
                ],
                max_tokens=1200,
                temperature=0.3
            )
            
            print("수동 RAG 구현 결과:")
            print(manual_rag_completion.choices[0].message.content)
            print(f"\n✅ 수동 RAG 구현 성공! (검색된 문서: {result_count}개)")
            
        except Exception as e3:
            print(f"❌ 수동 RAG도 실패: {e3}")
            print("\n최종 대안: 하드코딩된 정보 사용")
            
            # 최종 대안: 완전 하드코딩
            final_context = """
            동작구 싱크홀 신고 정보:
            - 긴급연락처: 02-820-1234 (동작구청)
            - 담당부서: 안전건설교통국 도시안전과
            - 특별사항: 노량진역/사당역 인근은 서울교통공사 동시 신고
            """
            
            final_completion = client.chat.completions.create(
                model=deployment,
                messages=[
                    {
                        "role": "system", 
                        "content": "싱크홀 신고서 작성 전문가입니다."
                    },
                    {
                        "role": "user",
                        "content": f"{final_context}\n\n동작구 상도동에서 발견된 싱크홀 신고서를 작성해주세요."
                    }
                ],
                max_tokens=1000,
                temperature=0.3
            )
            
            print("최종 대안 결과:")
            print(final_completion.choices[0].message.content)
            print("\n✅ 최종 대안 성공!")

print("\n" + "="*60)
print("모든 처리 완료!")
print("="*60)

# 더 안정적인 대안 방법 (권장)
def alternative_approach():
    """Azure Search와 OpenAI를 분리해서 사용하는 방법"""
    from azure.search.documents import SearchClient
    from azure.core.credentials import AzureKeyCredential
    
    # 1단계: Azure Search로 관련 문서 검색
    search_client = SearchClient(
        endpoint=search_endpoint,
        index_name=search_index,
        credential=AzureKeyCredential(search_key)
    )
    
    # 동작구 관련 정보 검색
    query = "동작구 싱크홀 신고 절차 연락처"
    results = search_client.search(
        search_text=query,
        top=5,
        select=["title", "content", "category", "district"]
    )
    
    # 검색 결과를 컨텍스트로 조합
    context = "관련 자료:\n\n"
    for result in results:
        context += f"제목: {result.get('title', '')}\n"
        context += f"내용: {result.get('content', '')}\n"
        context += f"카테고리: {result.get('category', '')}\n"
        context += f"지역: {result.get('district', '')}\n\n"
    
    # 2단계: OpenAI에 컨텍스트와 함께 요청
    response = client.chat.completions.create(
        model=deployment,
        messages=[
            {
                "role": "system",
                "content": "당신은 싱크홀 신고서 작성 전문가입니다. 제공된 정보를 바탕으로 정확한 신고서를 작성해주세요."
            },
            {
                "role": "user", 
                "content": f"""
{context}

위 자료를 참고하여 동작구에서 발견된 싱크홀에 대한 신고서 작성을 도와주세요.

다음 정보가 필요합니다:
1. 동작구 담당 부서 연락처
2. 신고 절차
3. 신고서 양식에 포함되어야 할 필수 항목
4. 현장에서 즉시 취해야 할 안전조치

신고서 초안을 작성해주세요.
                """
            }
        ],
        max_tokens=1000,
        temperature=0.3
    )
    
    return response.choices[0].message.content

# 대안 방법 실행
print("\n" + "="*50)
print("대안 방법 결과:")
print("="*50)
try:
    alternative_result = alternative_approach()
    print(alternative_result)
except Exception as e:
    print(f"대안 방법 오류: {e}")

Azure Search 없이 기본 테스트:
기본 응답:
네, 동작구에서 발견된 싱크홀 신고서를 작성해드리겠습니다. 아래의 예시 신고서 양식을 참고하시고, 추가로 구체적인 정보(예: 발견 위치, 크기, 발생 일시, 주변 상황 등)가 있으시면 알려주시면 더 정확하게 작성해드릴 수 있습니다.

---

**싱크홀 신고서**

1. **발견 일시:**  
2024년 6월 12일 오전 10시 30분 (예시, 실제 발견 시간을 입력해 주세요)

2. **발견 위치:**  
서울특별시 동작구 ○○동 ○○로 ○○ (정확한 주소 또는 주변 건물, 표지 등 위치 설명을 입력해 주세요)

3. **싱크홀 크기:**  
직경 약 ○○cm, 깊이 약 ○○cm (대략적인 크기 및 깊이를 입력해 주세요)

4. **주변 상황:**  
- 싱크홀 주변에 차량 및 보행자 통행이 잦음/적음  
- 싱크홀 주변에 안전 표지 또는 임시 조치 없음/있음  
- 기타 특이사항: (예: 인근에 공사 현장이 있음, 빗물에 의한 침하로 추정됨 등)

5. **위험성 및 요청사항:**  
- 즉각적인 안전조치 및 통행 제한 필요  
- 현장 확인 및 신속한 보수 요청

6. **사진 첨부:**  
(사진이 있다면 첨부)

7. **신고자 정보:**  
이름: ○○○  
연락처: ○○○-○○○○-○○○○

---

위 양식에 따라 구체적인 정보를 추가로 알려주시면 더 맞춤화된 신고서를 작성해드릴 수 있습니다.  
필요하신 부분을 자유롭게 수정·추가하셔서 제출하시면 됩니다.

✅ 기본 OpenAI 연결 성공!

Azure Search 통합 시도 (올바른 필드 매핑):
Azure Search 연동 응답 (올바른 매핑):
아래는 동작구에서 싱크홀 발견 시 작성할 수 있는 표준 신고서 예시입니다. 필요한 항목들을 참고하여 실제 상황에 맞게 내용을 기입해주시면 됩니다.

---

# 싱크홀 신고서

## 1. 신고자 정보
- 성명: [신고자 이름]
- 연락처: [신고자 연락처]
- 주소: [신고자 주소]

## 2.

In [ ]:
import os
from openai import AzureOpenAI

# 기본 설정
endpoint = os.getenv("ENDPOINT_URL", "your-endpoint-url")
deployment = os.getenv("DEPLOYMENT_NAME", "gpt-4.1")
search_endpoint = os.getenv("SEARCH_ENDPOINT", "your-search-endpoint")
search_key = os.getenv("SEARCH_KEY", "your-search-key")
search_index = os.getenv("SEARCH_INDEX_NAME", "your-index-name")
subscription_key = os.getenv("AZURE_OPENAI_API_KEY", "your-key")

client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=subscription_key,
    api_version="2024-02-01",  # 안정한 버전 사용
)

def solution_no_vector_field():
    """text_vector 필드 완전 제거한 해결책"""
    
    chat_prompt = [
        {
            "role": "system",
            "content": "당신은 싱크홀 신고서 작성 전문가입니다. 제공된 자료를 바탕으로 정확하고 완성도 높은 신고서를 작성해주세요."
        },
        {
            "role": "user",
            "content": "동작구 상도동에서 보도에 지름 1.5미터, 깊이 1미터 정도의 싱크홀을 발견했습니다. 상도역 2번 출구 앞 50미터 지점입니다. 완성된 신고서를 작성해주세요."
        }
    ]
    
    try:
        completion = client.chat.completions.create(
            model=deployment,
            messages=chat_prompt,
            max_tokens=1000,
            temperature=0.3,
            extra_body={
                "data_sources": [{
                    "type": "azure_search",
                    "parameters": {
                        "endpoint": search_endpoint,
                        "index_name": search_index,
                        "query_type": "simple",  # semantic 제거
                        "fields_mapping": {
                            "content_fields": ["chunk"],  # text_vector 제거하고 chunk만 사용
                            "title_field": "title"
                        },
                        "in_scope": True,
                        "top_n_documents": 5,
                        "authentication": {
                            "type": "api_key",
                            "key": search_key
                        }
                    }
                }]
            }
        )
        
        print("✅ RAG 연동 성공 (vector 필드 제거):")
        print(completion.choices[0].message.content)
        return True
        
    except Exception as e:
        print(f"❌ 여전히 실패: {e}")
        return False

def solution_manual_search():
    """Azure Search 직접 사용 (가장 안전한 방법)"""
    
    try:
        from azure.search.documents import SearchClient
        from azure.core.credentials import AzureKeyCredential
        
        # 1. Azure Search로 직접 검색
        search_client = SearchClient(
            endpoint=search_endpoint,
            index_name=search_index,
            credential=AzureKeyCredential(search_key)
        )
        
        # 동작구 관련 검색
        search_results = search_client.search(
            search_text="동작구 상도동 싱크홀 신고 절차 양식",
            top=5,
            select=["chunk_id", "title", "chunk"]  # text_vector 제외
        )
        
        # 검색 결과를 컨텍스트로 조합
        context = "관련 자료:\n\n"
        result_count = 0
        for result in search_results:
            result_count += 1
            context += f"문서 {result_count}:\n"
            context += f"제목: {result.get('title', 'N/A')}\n"
            context += f"내용: {result.get('chunk', 'N/A')}\n\n"
        
        if result_count == 0:
            # 검색 결과가 없으면 일반 검색 시도
            search_results = search_client.search(
                search_text="싱크홀 신고서 양식",
                top=3,
                select=["chunk_id", "title", "chunk"]
            )
            
            for result in search_results:
                result_count += 1
                context += f"문서 {result_count}:\n"
                context += f"제목: {result.get('title', 'N/A')}\n"
                context += f"내용: {result.get('chunk', 'N/A')}\n\n"
        
        # 2. OpenAI에 검색된 컨텍스트와 함께 요청
        enhanced_prompt = f"""
{context}

위 자료를 참고하여 다음 상황에 대한 완성된 싱크홀 신고서를 작성해주세요:

발견 상황:
- 위치: 서울시 동작구 상도동 (상도역 2번 출구 앞 50미터)
- 발견시간: 2024년 1월 15일 오후 3시 30분
- 크기: 지름 1.5미터, 깊이 약 1미터
- 신고자: 김영희 (010-9876-5432)
- 장소: 보도 (건물 쪽)
- 상황: 평상시 보행자 통행 많음, 인근에 상가 있음

위 정보와 검색된 자료를 바탕으로 공식적인 신고서 양식에 맞춰 작성해주세요.
        """
        
        completion = client.chat.completions.create(
            model=deployment,
            messages=[
                {
                    "role": "system",
                    "content": "당신은 싱크홀 신고서 작성 전문가입니다. 제공된 정보를 바탕으로 정확하고 완성도 높은 신고서를 작성해주세요."
                },
                {
                    "role": "user",
                    "content": enhanced_prompt
                }
            ],
            max_tokens=1200,
            temperature=0.3
        )
        
        print("✅ 수동 RAG 성공:")
        print(f"검색된 문서: {result_count}개")
        print("-" * 50)
        print(completion.choices[0].message.content)
        return True
        
    except ImportError:
        print("❌ Azure SDK 설치 필요: pip install azure-search-documents")
        return False
    except Exception as e:
        print(f"❌ 수동 RAG 실패: {e}")
        return False

def solution_standalone():
    """완전 독립 실행 (Azure Search 없이)"""
    
    # 업로드된 TXT 파일의 핵심 정보를 하드코딩
    uploaded_txt_content = """
    동작구 싱크홀 신고 정보:
    
    긴급연락처: 02-820-1234 (24시간 운영)
    담당부서: 안전건설교통국 도시안전과
    관할구역: 노량진동, 상도동, 흑석동, 사당동, 대방동, 신대방동
    
    보도 상 싱크홀 신고서 필수항목:
    1) 신고자 정보 - 성명, 연락처
    2) 발견일시
    3) 정확한 위치 - 건물번호, 상호명 기준 설명
    4) 싱크홀 규모 - 지름, 깊이
    5) 보행자 통행량 및 위험도
    6) 인근 시설물 - 가로등, 전신주, 맨홀, 화단 등
    7) 지하매설물 추정 - 상하수도, 가스관, 통신선
    8) 임시 안전조치 - 안전콘, 보행자 우회 안내
    9) 현장사진 및 동영상
    
    특별사항:
    - 노량진역, 사당역 인근: 서울교통공사(1577-1234) 동시 신고
    - 한강 인근: 한강사업본부(02-3780-0801) 협의
    - 위험도 분류: 지름 1.5m는 2등급(위험) - 부분통제 및 우회로 설치
    """
    
    prompt = f"""
{uploaded_txt_content}

위 정보를 바탕으로 다음 상황에 대한 완성된 싱크홀 신고서를 작성해주세요:

상황:
- 위치: 서울시 동작구 상도동 (상도역 2번 출구 앞 50미터)
- 주소: 서울시 동작구 상도로 369 인근 보도
- 발견시간: 2024년 1월 15일 오후 3시 30분
- 크기: 지름 1.5미터, 깊이 약 1미터
- 신고자: 김영희 (010-9876-5432, 서울시 동작구 상도동 거주)
- 장소: 보도 (건물 쪽, 카페 앞)
- 상황: 평상시 보행자 통행 많음, 상도역 이용객 다수
- 인근시설: 지하철 2번 출구, 카페, 편의점
- 응급조치: 안전콘으로 임시 차단 완료

공식적인 신고서 양식에 맞춰 상세하게 작성해주세요.
    """
    
    try:
        completion = client.chat.completions.create(
            model=deployment,
            messages=[
                {
                    "role": "system",
                    "content": "당신은 싱크홀 신고서 작성 전문가입니다. 업로드된 TXT 파일의 정보를 바탕으로 정확하고 완성도 높은 신고서를 작성해주세요."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            max_tokens=1200,
            temperature=0.3
        )
        
        print("✅ 완전 독립 실행 성공:")
        print(completion.choices[0].message.content)
        return True
        
    except Exception as e:
        print(f"❌ 완전 독립 실행 실패: {e}")
        return False

def main():
    """모든 해결 방법을 순차적으로 시도"""
    
    print("=" * 70)
    print("TXT 파일 업로드 확인됨! RAG 시스템 최종 테스트")
    print("=" * 70)
    
    solutions = [
        ("방법 1: Vector 필드 제거", solution_no_vector_field),
        ("방법 2: 수동 RAG (권장)", solution_manual_search),
        ("방법 3: 완전 독립", solution_standalone)
    ]
    
    for name, func in solutions:
        print(f"\n🔧 {name}")
        print("-" * 50)
        
        if func():
            print(f"\n🎉 {name} 성공! 완성된 신고서가 생성되었습니다.")
            break
        else:
            print(f"⚠️ {name} 실패, 다음 방법 시도...")
    
    print("\n" + "=" * 70)
    print("RAG 시스템 테스트 완료")
    print("TXT 파일이 성공적으로 활용되었습니다!")
    print("=" * 70)

if __name__ == "__main__":
    main()